# **Image Super Resolution — ESPCN**

**Updated for augmented dataset (149490 images)**



## **Cell 1 — Imports & Device**

In [ ]:
import os, random, math
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt

# setting up my main folders

PROJECT_ROOT = Path.home() / 'sr_project'
HR_DIR  = PROJECT_ROOT / 'HR_256'
LR_DIR  = PROJECT_ROOT / 'LR_x4'
OUT_DIR = PROJECT_ROOT / 'espcn_compare'
CKPT    = PROJECT_ROOT / 'best_espcn_x4.pth'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# just check if folder exist

assert HR_DIR.exists(), f'HR_256 not found at {HR_DIR}'
assert LR_DIR.exists(), f'LR_x4  not found at {LR_DIR}'

print('HR_DIR :', HR_DIR)
print('LR_DIR :', LR_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

HR_DIR : /home/rokib/sr_project/HR_256
LR_DIR : /home/rokib/sr_project/LR_x4

Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Ti
VRAM   : 17.1 GB


## **Cell 2 — Auto-Detect Dataset Size & Build Train/Val/Test IDs**

This code now scan the HR folder and split **80/10/10** automatic.


In [ ]:
# ── Scan HR_256 and collect all image stems (the numeric IDs) ─────────────────
all_stems = sorted(
    p.stem for p in HR_DIR.iterdir()
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg') and
       (LR_DIR / p.name).exists()   # only keep IDs that have a matching LR file
)

total = len(all_stems)
assert total >= 10, f'Too few images found: {total}. Check HR_256 and LR_x4 folders.'

# calculate 80 / 10 / 10 split
n_train = int(total * 0.80)
n_val   = int(total * 0.10)
n_test  = total - n_train - n_val

# random shuffle with seed 42 so it stay same every run
rng = random.Random(42)
shuffled = all_stems.copy()
rng.shuffle(shuffled)

train_ids = shuffled[:n_train]
val_ids   = shuffled[n_train:n_train + n_val]
test_ids  = shuffled[n_train + n_val:]

print(f'Total paired images : {total}')
print(f'Train               : {len(train_ids)}  ({len(train_ids)/total*100:.0f}%)')
print(f'Val                 : {len(val_ids)}   ({len(val_ids)/total*100:.0f}%)')
print(f'Test                : {len(test_ids)}   ({len(test_ids)/total*100:.0f}%)')

Total paired images : 149490
Train               : 119592  (80%)
Val                 : 14949   (10%)
Test                : 14949   (10%)


## **Cell 3 — Dataset Classes**

In [ ]:
def find_img(folder: Path, id_: str) -> Path:
    for ext in ['.png', '.jpg', '.jpeg']:
        p = folder / f'{id_}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No image for id={id_} in {folder}')


class SRDataset(Dataset):
    """dataset for training. it load patches and apply some augmentation"""
    def __init__(self, lr_dir, hr_dir, ids, scale=4, patch_size=32, augment=True):
        self.lr_dir     = Path(lr_dir)
        self.hr_dir     = Path(hr_dir)
        self.ids        = ids
        self.scale      = scale
        self.patch_size = patch_size
        self.augment    = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        lr  = TF.to_tensor(Image.open(find_img(self.lr_dir, id_)).convert('RGB'))
        hr  = TF.to_tensor(Image.open(find_img(self.hr_dir, id_)).convert('RGB'))

        _, h, w = lr.shape
        p, s    = self.patch_size, self.scale

        # check if LR image is smaller than patch. if yes then skip the random crop
        if h <= p or w <= p:
            lr_patch = TF.resize(lr, [p, p])
            hr_patch = TF.resize(hr, [p*s, p*s])
        else:
            top  = random.randint(0, h - p)
            left = random.randint(0, w - p)
            lr_patch = lr[:, top:top+p,       left:left+p]
            hr_patch = hr[:, top*s:(top+p)*s, left*s:(left+p)*s]

        if self.augment:
            if random.random() < 0.5:
                lr_patch = TF.hflip(lr_patch); hr_patch = TF.hflip(hr_patch)
            if random.random() < 0.5:
                lr_patch = TF.vflip(lr_patch); hr_patch = TF.vflip(hr_patch)
            k = random.randint(0, 3)
            if k:
                lr_patch = torch.rot90(lr_patch, k, [1, 2])
                hr_patch = torch.rot90(hr_patch, k, [1, 2])

        return lr_patch, hr_patch


class SRImageDataset(Dataset):
    """Full-image dataset for validation / test."""
    def __init__(self, lr_dir, hr_dir, ids):
        self.lr_dir = Path(lr_dir)
        self.hr_dir = Path(hr_dir)
        self.ids    = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        lr  = TF.to_tensor(Image.open(find_img(self.lr_dir, id_)).convert('RGB'))
        hr  = TF.to_tensor(Image.open(find_img(self.hr_dir, id_)).convert('RGB'))
        return lr, hr, id_


print('Dataset classes defined.')

Dataset classes defined.


## **Cell 4 — DataLoaders**

In [4]:
SCALE = 4
PATCH = 32
BATCH = 16

train_ds     = SRDataset(LR_DIR, HR_DIR, train_ids, scale=SCALE, patch_size=PATCH, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

val_ds       = SRImageDataset(LR_DIR, HR_DIR, val_ids)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

test_ds      = SRImageDataset(LR_DIR, HR_DIR, test_ids)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

print(f'Train  samples  : {len(train_ds)}  →  {len(train_loader)} batches (batch_size={BATCH})')
print(f'Val    images   : {len(val_ds)}')
print(f'Test   images   : {len(test_ds)}')

Train  samples  : 119592  →  7475 batches (batch_size=16)
Val    images   : 14949
Test   images   : 14949
